In [1]:
import pandas as pd
import re

def parse_pepsi_income_statement(data_text):
    """
    解析百事可乐利润表数据
    """
    # 按行分割数据
    tokens = data_text.strip().split(' ')
    tokens = re.split(r'[\s\$\s|\s]', data_text.strip())
    lines = []
    temp = []
    for i in range(len(tokens)):
        temp.append(tokens[i].strip())
        if i % 4 == 0:
            lines.append(temp)
            temp = []
    
    # 初始化数据结构
    data = []
    current_category = ""
    
    for line in lines:
        line = ' '.join(line).strip()
        # 检查是否是分类行（不含数字的行）
        if not any(char.isdigit() for char in line):
            current_category = line
            continue
            
        # 解析数据行
        parts = re.split(r'\s{2,}', line)  # 按多个空格分割
        if len(parts) >= 4:
            account_name = parts[0]
            values_2024 = parts[1].replace('$', '').replace(',', '').strip()
            values_2023 = parts[2].replace('$', '').replace(',', '').strip()
            values_2022 = parts[3].replace('$', '').replace(',', '').strip()
            
            # 处理空值
            values_2024 = '0' if values_2024 == '—' else values_2024
            values_2023 = '0' if values_2023 == '—' else values_2023
            values_2022 = '0' if values_2022 == '—' else values_2022
            
            data.append({
                'Category': current_category,
                'Account': account_name,
                '2024': float(values_2024) if values_2024 else 0,
                '2023': float(values_2023) if values_2023 else 0,
                '2022': float(values_2022) if values_2022 else 0
            })
    
    return data

def create_excel_output(parsed_data, filename='pepsi_income_statement.xlsx'):
    """
    创建Excel输出
    """
    # 创建DataFrame
    df = pd.DataFrame(parsed_data)
    
    # 重新排列列的顺序
    df = df[['Category', 'Account', '2022', '2023', '2024']]
    
    # 创建Excel writer对象
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # 主要数据表
        df.to_excel(writer, sheet_name='Income Statement', index=False)
        
        # 创建转置版本便于阅读
        transposed_df = df.set_index(['Category', 'Account']).T
        transposed_df.to_excel(writer, sheet_name='Transposed View')
        
        # 创建关键指标汇总表
        key_metrics = [
            'Net Revenue',
            'Cost of sales', 
            'Gross profit',
            'Operating Profit',
            'Net income',
            'Net Income Attributable to PepsiCo'
        ]
        
        key_data = []
        for metric in key_metrics:
            for item in parsed_data:
                if metric in item['Account']:
                    key_data.append({
                        'Metric': item['Account'],
                        '2022': item['2022'],
                        '2023': item['2023'], 
                        '2024': item['2024']
                    })
                    break
        
        key_df = pd.DataFrame(key_data)
        key_df.to_excel(writer, sheet_name='Key Metrics', index=False)
    
    # 调整列宽
    from openpyxl import load_workbook
    wb = load_workbook(filename)
    
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for column in ws.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = (max_length + 2)
            ws.column_dimensions[column_letter].width = adjusted_width
    
    wb.save(filename)
    return df

# 你的数据
pepsi_data = """Net Revenue $ 91,854 $ 91,471 $ 86,392 Cost of sales 41,744 41,881 40,576 Gross profit 50,110 49,590 45,816 Selling, general and administrative expenses 37,190 36,677 34,459 Gain associated with the Juice Transaction (see Note 13) — — (3,321) Impairment of intangible assets (see Notes 1 and 4) 33 927 3,166 Operating Profit 12,887 11,986 11,512 Other pension and retiree medical benefits (expense)/income (22) 250 132 Net interest expense and other (919) (819) (939) Income before income taxes 11,946 11,417 10,705 Provision for income taxes 2,320 2,262 1,727 Net income 9,626 9,155 8,978 Less: Net income attributable to noncontrolling interests 48 81 68 Net Income Attributable to PepsiCo $ 9,578 $ 9,074 $ 8,910 Net Income Attributable to PepsiCo per Common Share Basic $ 6.97 $ 6.59 $ 6.45 Diluted $ 6.95 $ 6.56 $ 6.42 Weighted-average common shares outstanding Basic 1,373 1,376 1,380 Diluted 1,378 1,383 1,387"""

# 执行解析和导出
print("正在解析百事可乐利润表数据...")
parsed_data = parse_pepsi_income_statement(pepsi_data)

print("正在生成Excel文件...")
result_df = create_excel_output(parsed_data, 'pepsi_income_statement_2022-2024.xlsx')

print("完成！已生成文件: pepsi_income_statement_2022-2024.xlsx")
print("\n解析后的数据预览:")
print(result_df.to_string(index=False))

# 显示关键指标
print("\n关键指标 (单位: 百万美元):")
key_items = ['Net Revenue', 'Gross profit', 'Operating Profit', 'Net Income Attributable to PepsiCo']
for item in parsed_data:
    if any(key in item['Account'] for key in key_items):
        print(f"{item['Account']}: {item['2022']} | {item['2023']} | {item['2024']}")

正在解析百事可乐利润表数据...
正在生成Excel文件...


KeyError: "None of [Index(['Category', 'Account', '2022', '2023', '2024'], dtype='object')] are in the [columns]"

In [2]:
import pandas as pd
import re

def parse_pepsi_income_statement_fixed(data_text):
    """
    修复版的百事可乐利润表数据解析
    """
    print("开始解析数据...")
    
    # 按行分割数据
    lines = data_text.replace('$', '').replace(',', '').replace('—', '0').strip('\n')
    parsed_lines = []

    for line in lines:
        parsed_lines.append(re.split(r'[\w\s|\d)\s\w(?!\())]', line.strip()));
    
    # 初始化数据结构
    data = []
    current_category = ""
    
    for i, line in enumerate(lines):
        line = ' '.join(line).strip()
        line = line.strip()
        print(f"处理第{i}行: {line[:50]}...")  # 调试信息
        
        if not line:
            continue
            
        # 检查是否是分类行（不含数字的行）
        line_clean = line.replace('$', '').replace(',', '').replace('—', '').replace('(', '').replace(')', '')
        if not any(char.isdigit() for char in line_clean):
            current_category = line
            print(f"  设置为分类: {current_category}")
            continue
            
        # 尝试多种解析方式
        parts = None
        
        # 方式1: 按多个空格分割
        parts = re.split(r'\s{2,}', line)
        if len(parts) < 4:
            # 方式2: 按数字前的空格分割
            parts = re.split(r'\s+(?=\$?\d)', line)
        
        if len(parts) < 4:
            # 方式3: 简单的空格分割
            parts = line.split()
        
        print(f"  解析出 {len(parts)} 个部分: {parts}")
            
        if len(parts) >= 4:
            # 重建账户名（前几个非数字部分）
            account_parts = []
            value_parts = []
            
            for part in parts:
                part_clean = part.replace('$', '').replace(',', '').replace('—', '0').strip()
                # 检查是否是数值
                if re.match(r'^-?\(?\d+\.?\d*\)?$', part_clean.replace('(', '').replace(')', '')):
                    value_parts.append(part)
                else:
                    account_parts.append(part)
            
            if len(value_parts) >= 3:
                account_name = ' '.join(account_parts)
                values_2024 = value_parts[0].replace('$', '').replace(',', '').replace('—', '0').replace('(', '-').replace(')', '').strip()
                values_2023 = value_parts[1].replace('$', '').replace(',', '').replace('—', '0').replace('(', '-').replace(')', '').strip()
                values_2022 = value_parts[2].replace('$', '').replace(',', '').replace('—', '0').replace('(', '-').replace(')', '').strip()
                
                # 处理空值和负值
                try:
                    val_2024 = float(values_2024) if values_2024 and values_2024 != '—' else 0.0
                    val_2023 = float(values_2023) if values_2023 and values_2023 != '—' else 0.0
                    val_2022 = float(values_2022) if values_2022 and values_2022 != '—' else 0.0
                except ValueError as e:
                    print(f"  数值转换错误: {e}, 使用默认值0")
                    val_2024 = val_2023 = val_2022 = 0.0
                
                item_data = {
                    'Category': current_category,
                    'Account': account_name,
                    '2022': val_2022,
                    '2023': val_2023,
                    '2024': val_2024
                }
                
                data.append(item_data)
                print(f"  成功解析: {account_name} -> {val_2022}, {val_2023}, {val_2024}")
            else:
                print(f"  数值部分不足3个，跳过此行")
        else:
            print(f"  部分数量不足，跳过此行")
    
    print(f"解析完成，共找到 {len(data)} 条记录")
    return data

def create_excel_output_fixed(parsed_data, filename='pepsi_income_statement.xlsx'):
    """
    修复版的Excel输出函数
    """
    if not parsed_data:
        print("错误: 没有数据可导出!")
        return None
    
    print("创建DataFrame...")
    # 创建DataFrame
    df = pd.DataFrame(parsed_data)
    
    print("DataFrame列名:", df.columns.tolist())
    print("DataFrame形状:", df.shape)
    
    # 检查必要的列是否存在
    required_columns = ['Category', 'Account', '2022', '2023', '2024']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"警告: 缺少列 {missing_columns}")
        print("可用列:", df.columns.tolist())
        return None
    
    # 重新排列列的顺序
    df = df[['Category', 'Account', '2022', '2023', '2024']]
    
    try:
        # 创建Excel writer对象
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # 主要数据表
            df.to_excel(writer, sheet_name='Income Statement', index=False)
            print("主要数据表导出完成")
            
            # 创建转置版本便于阅读
            try:
                transposed_df = df.set_index(['Category', 'Account']).T
                transposed_df.to_excel(writer, sheet_name='Transposed View')
                print("转置视图导出完成")
            except Exception as e:
                print(f"转置视图导出失败: {e}")
            
            # 创建关键指标汇总表
            try:
                key_metrics = [
                    'Net Revenue',
                    'Cost of sales', 
                    'Gross profit',
                    'Operating Profit',
                    'Net income',
                    'Net Income Attributable to PepsiCo'
                ]
                
                key_data = []
                for metric in key_metrics:
                    for item in parsed_data:
                        if metric.lower() in item['Account'].lower():
                            key_data.append({
                                'Metric': item['Account'],
                                '2022': item['2022'],
                                '2023': item['2023'], 
                                '2024': item['2024']
                            })
                            break
                
                if key_data:
                    key_df = pd.DataFrame(key_data)
                    key_df.to_excel(writer, sheet_name='Key Metrics', index=False)
                    print("关键指标表导出完成")
                
            except Exception as e:
                print(f"关键指标表导出失败: {e}")
        
        print(f"Excel文件 '{filename}' 创建成功")
        
    except Exception as e:
        print(f"Excel导出错误: {e}")
        return None
    
    return df

# 测试数据 - 使用更简单的格式进行测试
test_data = """Net Revenue
$ 91,854 $ 91,471 $ 86,392
Cost of sales
41,744 41,881 40,576
Gross profit
50,110 49,590 45,816
Operating Profit
12,887 11,986 11,512
Net income
9,626 9,155 8,978
Net Income Attributable to PepsiCo
$ 9,578 $ 9,074 $ 8,910"""

# 你的完整数据
pepsi_data = """
Net Revenue $ 91,854 $ 91,471 $ 86,392
Cost of sales 41,744 41,881 40,576
Gross profit 50,110 49,590 45,816
Selling, general and administrative expenses 37,190 36,677 34,459
Gain associated with the Juice Transaction (see Note 13) — — (3,321)
Impairment of intangible assets (see Notes 1 and 4) 33 927 3,166
Operating Profit 12,887 11,986 11,512 Other pension and retiree medical benefits (expense)/income (22) 250 132
Net interest expense and other (919) (819) (939)
Income before income taxes 11,946 11,417 10,705
Provision for income taxes 2,320 2,262 1,727
Net income 9,626 9,155 8,978
Less: Net income attributable to noncontrolling interests 48 81 68
Net Income Attributable to PepsiCo $ 9,578 $ 9,074 $ 8,910
Net Income Attributable to PepsiCo per Common Share Basic $ 6.97 $ 6.59 $ 6.45
Diluted $ 6.95 $ 6.56 $ 6.42
Weighted-average common shares outstanding Basic 1,373 1,376 1,380
Diluted 1,378 1,383 1,387
"""

print("=== 开始解析百事可乐利润表数据 ===")

# 先用测试数据验证
print("\n1. 使用测试数据验证...")
test_parsed = parse_pepsi_income_statement_fixed(test_data)

if test_parsed:
    print("\n测试数据解析成功!")
    test_df = create_excel_output_fixed(test_parsed, 'test_pepsi_income_statement.xlsx')
    if test_df is not None:
        print("\n测试数据预览:")
        print(test_df.to_string(index=False))
else:
    print("测试数据解析失败!")

# 再用完整数据
print("\n2. 使用完整数据...")
parsed_data = parse_pepsi_income_statement_fixed(pepsi_data)

if parsed_data:
    print("\n完整数据解析成功!")
    result_df = create_excel_output_fixed(parsed_data, 'pepsi_income_statement_2022-2024.xlsx')
    
    if result_df is not None:
        print("\n解析后的数据预览:")
        print(result_df.to_string(index=False))
        
        # 显示关键指标
        print("\n关键指标 (单位: 百万美元):")
        key_items = ['Net Revenue', 'Gross profit', 'Operating Profit', 'Net Income Attributable to PepsiCo']
        for item in parsed_data:
            if any(key in item['Account'] for key in key_items):
                print(f"{item['Account']}: {item['2022']} | {item['2023']} | {item['2024']}")
else:
    print("完整数据解析失败!")

=== 开始解析百事可乐利润表数据 ===

1. 使用测试数据验证...
开始解析数据...
处理第0行: Net...
  设置为分类: Net
处理第1行: Revenue   91,854...
  解析出 2 个部分: ['Revenue', '91,854']
  部分数量不足，跳过此行
处理第2行: 91,471...
  解析出 1 个部分: ['91,471']
  部分数量不足，跳过此行
处理第3行: 86,392 Cost of...
  解析出 3 个部分: ['86,392', 'Cost', 'of']
  部分数量不足，跳过此行
处理第4行: sales 41,744 41,881 40,576...
  解析出 4 个部分: ['sales', '41,744', '41,881', '40,576']
  成功解析: sales -> 40576.0, 41881.0, 41744.0
处理第5行: Gross profit 50,110 49,590...
  解析出 4 个部分: ['Gross', 'profit', '50,110', '49,590']
  数值部分不足3个，跳过此行
处理第6行: 45,816 Operating Profit 12,887...
  解析出 4 个部分: ['45,816', 'Operating', 'Profit', '12,887']
  数值部分不足3个，跳过此行
处理第7行: 11,986 11,512 Net income...
  解析出 4 个部分: ['11,986', '11,512', 'Net', 'income']
  数值部分不足3个，跳过此行
处理第8行: 9,626 9,155 8,978 Net...
  解析出 4 个部分: ['9,626', '9,155', '8,978', 'Net']
  成功解析: Net -> 8978.0, 9155.0, 9626.0
处理第9行: Income Attributable to PepsiCo...
  设置为分类: Income Attributable to PepsiCo
处理第10行: 9,578...
  解析出 1 个部分: ['9,578']
  部分数量不足，跳过此行
处理第11行: 9

In [2]:
import re
from textwrap import dedent

raw_data = dedent("""
	Coca					Pepsi		
	2022	2023	2024			2022	2023	2024
a)     Return on assets	0.10 	0.11 	0.11 			0.10 	0.10 	0.10 
	9571.00 	10703.00 	10649.00 			8978.00 	9155.00 	9626.00 
	93558.50 	95233.00 	99126.00 			92282.00 	96341.00 	99981.00 
								
b)     Debt ratio 	0.72 	0.72 	0.74 			0.81 	0.81 	0.82 
	66937.00 	70223.00 	74177.00 			74914.00 	81858.00 	81296.00 
	92763.00 	97703.00 	100549.00 			92187.00 	100495.00 	99467.00 
								
c)     Net Profit margin	0.22 	0.23 	0.23 			0.10 	0.10 	0.10 
	9571.00 	10703.00 	10649.00 			8978.00 	9155.00 	9626.00 
	43004.00 	45754.00 	47061.00 			86392.00 	91471.00 	91854.00 
								
d)     Current ratio	1.15 	1.13 	1.03 			0.80 	0.85 	0.82 
	22591.00 	26732.00 	25997.00 			21539.00 	26950.00 	25826.00 
	19724.00 	23571.00 	25249.00 			26785.00 	31647.00 	31536.00 
								
e)     Quick ratio(?)	0.77 	0.72 	0.72 			0.58 	0.66 	0.62 
	15118.00 	17073.00 	18140.00 			15511.00 	20818.00 	19599.00 
	19724.00 	23571.00 	25249.00 			26785.00 	31647.00 	31536.00 
								
f)     Gross profit percentage	0.58 	0.60 	0.61 			0.53 	0.54 	0.55 
	25004.00 	27234.00 	28737.00 			45816.00 	49590.00 	50110.00 
	43004.00 	45754.00 	47061.00 			86392.00 	91471.00 	91854.00 
								
g)     Inventory turnover ratio	4.71 	4.28 	4.00 			8.48 	7.94 	7.85 
	18000.00 	18520.00 	18324.00 			40576.00 	41881.00 	41744.00 
	3823.50 	4328.50 	4576.00 			4784.50 	5278.00 	5320.00 
								
h)     Average days to sell inventory	77.49 	85.28 	91.25 			43.04 	45.97 	46.50 
								
								
								
i)     Receivable turnover ratio	12.29 	13.27 	13.49 			8.50 	8.72 	8.69 
	43004.00 	45754.00 	47061.00 			86392.00 	91471.00 	91854.00 
	3499.50 	3448.50 	3489.50 			10163.00 	10489.00 	10574.00 
								
j)     Total asset turnover ratio	0.46 	0.48 	0.47 			0.94 	0.95 	0.92 
	43004.00 	45754.00 	47061.00 			86392.00 	91471.00 	91854.00 
	93558.50 	95233.00 	99126.00 			92282.00 	96341.00 	99981.00 
								
k)     Times interest earned ratio	14.25 	9.48 	8.90 			10.57 	8.95 	8.44 
	12568.00 	14479.00 	14742.00 			11824.00 	12854.00 	13552.00 
	882.00 	1527.00 	1656.00 			1119.00 	1437.00 	1606.00 
								
l)     Debt-to-equity ratio	2.59 	2.56 	2.81 			4.34 	4.39 	4.47 
	66937.00 	70223.00 	74177.00 			74914.00 	81858.00 	81296.00 
	25826.00 	27480.00 	26372.00 			17273.00 	18637.00 	18171.00 
								
m)    Book value per ordinary share	5.57 	6.02 	5.78 			12.45 	13.47 	13.15 
	24105.00 	25941.00 	24856.00 			17149.00 	18503.00 	18041.00 
	4328.00 	4308.00 	4302.00 			1377.00 	1374.00 	1372.00 
								
n)     Basic earnings per share (EPS)	2.20 	2.48 	2.47 			6.46 	6.59 	6.98 
	9542.00 	10714.00 	10631.00 			8910.00 	9074.00 	9578.00 
	4328.00 	4323.00 	4309.00 			1380.00 	1376.00 	1373.00 
								
o)    Price-earnings ratio (P/E ratio)	28.85 	23.78 	25.24 			27.98 	25.75 	21.92 
	63.61 	58.93 	62.26 			180.66 	169.84 	152.89 
	2.20 	2.48 	2.47 			6.46 	6.59 	6.98 
								
p)     Dividend yield ratio	0.03 	0.03 	0.03 			0.03 	0.03 	0.03 
	1.76 	1.84 	1.94 			4.53 	4.95 	5.33 
	63.61 	58.93 	62.26 			180.66 	169.84 	152.89 
								
q)     Cash flow on total assets	0.12 	0.12 	0.07 			0.12 	0.14 	0.13 
	11018.00 	11599.00 	6805.00 			10811.00 	13442.00 	12507.00 
	93558.50 	95233.00 	99126.00 			92282.00 	96341.00 	99981.00 
""").strip()


def parse_numbers(line: str):
    tokens = re.split(r'\s+', line.strip())
    nums = []
    for t in tokens:
        t = t.replace(',', '')
        try:
            nums.append(float(t))
        except ValueError:
            pass
    return nums


lines = raw_data.splitlines()
# print(lines)
metrics = {}
order = []

i = 0
while i < len(lines):
    raw_line = lines[i]
    line = raw_line.strip()
    m = re.match(r'^([a-z])\)', line)
    if m:
        key = m.group(1)
        order.append(key)

        # 下一行：结果行（ratio）
        result_line = lines[i] if i < len(lines) else ""

        # 结果行后两行（如果存在）：分子、分母
        numer_line = lines[i + 1] if i + 1 < len(lines) else ""
        denom_line = lines[i + 2] if i + 2 < len(lines) else ""
        # print(denom_line)

        numer = parse_numbers(numer_line)
        denom = parse_numbers(denom_line)

        metrics[key] = {
            "result": parse_numbers(result_line),
            "numer": numer,
            "denom": denom,
        }

        i += 3
    else:
        i += 1

# print(metrics)
# print(order)
# h：分子用 365，分母用 g 的结果行（ratio）
if 'h' in metrics and 'g' in metrics:
    metrics['h']['numer'] = [365.0] * 6
    metrics['h']['denom'] = metrics['g']['result']


def trim_trailing_zeros(x: float):
    if isinstance(x, float) and x.is_integer():
        return int(x)
    return x

result = []
# 按指标顺序、公司一级年份二级顺序，直接打印分子 / 分母
for key in order:
    numer = metrics[key]['numer']
    denom = metrics[key]['denom']
    # print(numer, denom)
    if len(numer) != 6 or len(denom) != 6:
        # 像 h 原本没有分子分母，这里如果没被正确填充就跳过
        continue
    for n, d in zip(numer, denom):
        result.append(trim_trailing_zeros(n))
        result.append(trim_trailing_zeros(d))
        # print(trim_trailing_zeros(n))
        # print(trim_trailing_zeros(d))
# print(result)

a = '''
10.23%	11.24%	10.74%			9.73%	9.50%	9.63%
 0.72 	 0.72 	 0.74 			 0.81 	 0.81 	 0.82 
22.26%	23.39%	22.63%			10.39%	10.01%	10.48%
 1.15 	 1.13 	 1.03 			 0.80 	 0.85 	 0.82 
 0.77 	 0.72 	 0.72 			 0.58 	 0.66 	 0.62 
58.14%	59.52%	61.06%			53.03%	54.21%	54.55%
 4.71 	 4.28 	 4.00 			 8.48 	 7.94 	 7.85 
 77.49 	 85.28 	 91.25 			 43.04 	 45.97 	 46.50 
 12.29 	 13.27 	 13.49 			 8.50 	 8.72 	 8.69 
 0.46 	 0.48 	 0.47 			 0.94 	 0.95 	 0.92 
 14.25 	 9.48 	 8.90 			 10.57 	 8.95 	 8.44 
 2.59 	 2.56 	 2.81 			 4.34 	 4.39 	 4.47 
 5.57 	 6.02 	 5.78 			 12.45 	 13.47 	 13.15 
 2.20 	 2.48 	 2.47 			 6.46 	 6.59 	 6.98 
 28.85 	 23.78 	 25.24 			 27.98 	 25.75 	 21.92 
2.77%	3.12%	3.12%			2.50%	2.91%	3.49%
11.78%	12.18%	6.87%			11.72%	13.95%	12.51%
'''
result2 = []
lines = a.strip().splitlines()
for line in lines:
    tokens = line.split()
    result2.extend(tokens)
# print(result2)

j = 0
for i in range(len(result)):
    print(result[i])
    if (i + 1) % 2 == 0:
        print(result2[j].strip('%'))
        j+=1



9571
93558.5
10.23
10703
95233
11.24
10649
99126
10.74
8978
92282
9.73
9155
96341
9.50
9626
99981
9.63
66937
92763
0.72
70223
97703
0.72
74177
100549
0.74
74914
92187
0.81
81858
100495
0.81
81296
99467
0.82
9571
43004
22.26
10703
45754
23.39
10649
47061
22.63
8978
86392
10.39
9155
91471
10.01
9626
91854
10.48
22591
19724
1.15
26732
23571
1.13
25997
25249
1.03
21539
26785
0.80
26950
31647
0.85
25826
31536
0.82
15118
19724
0.77
17073
23571
0.72
18140
25249
0.72
15511
26785
0.58
20818
31647
0.66
19599
31536
0.62
25004
43004
58.14
27234
45754
59.52
28737
47061
61.06
45816
86392
53.03
49590
91471
54.21
50110
91854
54.55
18000
3823.5
4.71
18520
4328.5
4.28
18324
4576
4.00
40576
4784.5
8.48
41881
5278
7.94
41744
5320
7.85
365
4.71
77.49
365
4.28
85.28
365
4
91.25
365
8.48
43.04
365
7.94
45.97
365
7.85
46.50
43004
3499.5
12.29
45754
3448.5
13.27
47061
3489.5
13.49
86392
10163
8.50
91471
10489
8.72
91854
10574
8.69
43004
93558.5
0.46
45754
95233
0.48
47061
99126
0.47
86392
92282
0.94
91471
9634

In [ ]:
a = '''
10.23%	11.24%	10.74%			9.73%	9.50%	9.63%
 0.72 	 0.72 	 0.74 			 0.81 	 0.81 	 0.82 
22.26%	23.39%	22.63%			10.39%	10.01%	10.48%
 1.15 	 1.13 	 1.03 			 0.80 	 0.85 	 0.82 
 0.77 	 0.72 	 0.72 			 0.58 	 0.66 	 0.62 
58.14%	59.52%	61.06%			53.03%	54.21%	54.55%
 4.71 	 4.28 	 4.00 			 8.48 	 7.94 	 7.85 
 77.49 	 85.28 	 91.25 			 43.04 	 45.97 	 46.50 
 12.29 	 13.27 	 13.49 			 8.50 	 8.72 	 8.69 
 0.46 	 0.48 	 0.47 			 0.94 	 0.95 	 0.92 
 14.25 	 9.48 	 8.90 			 10.57 	 8.95 	 8.44 
 2.59 	 2.56 	 2.81 			 4.34 	 4.39 	 4.47 
 5.57 	 6.02 	 5.78 			 12.45 	 13.47 	 13.15 
 2.20 	 2.48 	 2.47 			 6.46 	 6.59 	 6.98 
 28.85 	 23.78 	 25.24 			 27.98 	 25.75 	 21.92 
2.77%	3.12%	3.12%			2.50%	2.91%	3.49%
11.78%	12.18%	6.87%			11.72%	13.95%	12.51%
'''
